# Siting First Aid Stations with LSCP on Toronto's University Campus

*Authors:* [Timothy Ellersiek](https://github.com/TimMcCauley), [James Gaboardi](https://github.com/jGaboardi), [Germano Barcelos](https://github.com/gegen07)

This notebook implements the location set covering problem (LSCP) to site first aid stations for Toronto's University campus. LSCP determines locations such that as many demand points as possible are within the maximum service radius. By doing so, the number of locations required to cover all demand points is minimized. In contrast to the maximum covering location problem it does not require a maximum amount of locations as an input parameter. Hence, the use cases for LSCP are specific. One common scenario is finding suitable locations for hospitals in cities where the amount is governed by reachability preconditions to households in a certain amount of time.  

In this notebook demand is described as buildings within Toronto's University campus. We use [OSMnx](https://osmnx.readthedocs.io/) to fetch building and road geometries from [OpenStreetMap](https://openstreetmap.org) via the Overpass API.

The routing matrix is computed using [Open Source Routing Machine's demo server API](http://project-osrm.org).

## Prerequisites

To run this jupyter notebook locally, you will have to install the following additional libraries into your python environment.

- [folium](https://python-visualization.github.io/folium)
- [osmnx](https://osmnx.readthedocs.io/)
- [routing-py](https://github.com/gis-ops/routing-py)
- [shapely](https://github.com/shapely/shapely)

In [ ]:
%config InlineBackend.figure_format = "retina"
%load_ext watermark
%watermark

In [ ]:
import folium
import numpy
import osmnx
import pandas
import pulp
from routingpy import OSRM

from spopt.locate import LSCP

%watermark -w
%watermark -iv

--------------------------

Find the University of Toronto St. George campus boundary with OSMnx.

In [ ]:
campus = osmnx.geocode_to_gdf("University of Toronto, Toronto, Ontario, Canada")
campus_polygon = campus.geometry.iloc[0]

First of all we want to fetch all buildings in the University of Toronto boundary

In [ ]:
result = osmnx.features_from_polygon(campus_polygon, tags={"building": "university"})
result = result[result.index.get_level_values("element") == "way"]

In [ ]:
result.explore(tooltip="name")

And now we want to fetch some roads in the same area and return their center points which we can use for our potential locations for first aid stations

In [ ]:
result_roads = osmnx.features_from_polygon(campus_polygon, tags={"highway": True})
result_roads = result_roads[
    (result_roads.index.get_level_values("element") == "way")
    & result_roads["highway"].str.contains("secondary|tertiary|unclassified", na=False)
]
first_aid_stations_locations = result_roads.representative_point()

In [ ]:
m = result.explore()
first_aid_stations_locations.explore(m=m, color="black", tooltip=False)

## Computing the Distance (Routing) Matrix

Given our potential first aid station locations and the university buildings on the campus we want to compute our distance matrix. Instead of using simple euclidean distance we can make use of proper routes following roads which will yield more realistic travel times. The [routing-py library](https://github.com/gis-ops/routing-py) helps us achieve this which features the ability to use different open source frameworks such as OSRM or Valhalla as well as 3rd party API's such as Google Maps or HERE.

In [ ]:
client = OSRM(base_url="https://router.project-osrm.org")

Isolate all location coordinates

In [ ]:
all_locations = pandas.concat([result.representative_point(), first_aid_stations_locations])
all_locations = [[row.x, row.y] for row in all_locations.get_coordinates().itertuples()]
all_locations[:5]

Define source and target indices

In [ ]:
source_indices = list(range(len(result)))
target_indices = list(range(len(result), len(all_locations)))

In [ ]:
source_indices[-5:], target_indices[:5]

Generate routing matrix object

In [ ]:
osrm_routing_matrix = client.matrix(
    dry_run=False,
    locations=all_locations,
    sources=source_indices,
    destinations=target_indices,
    profile="foot",
)

Isolate route durations

In [ ]:
cost_matrix = numpy.array(osrm_routing_matrix.durations)
cost_matrix.shape

In [ ]:
cost_matrix

## Solving the LSCP

Last but not least we want to run the solver. We choose the `pulp.COIN_CMD` and set the distance cutoff to 4 minutes and 50 seconds afoot. Note that, the service radius is large since it is the minimum acceptable value that covers all buildings. To decrease this value, there are two options: remove buildings far away from aid stations or add more potential aid stations in sparse regions.

Finally we loop over the solution and add the sited stations and the corresponding allocated buildings to the map with different radiuses.

In [ ]:
# maximum acceptable service duration, 4 minutes and 50 seconds
SERVICE_RADIUS = 290

# set the solver (for available solvers run ``pulp.listSolvers(onlyAvailable=True)``)
solver = pulp.COIN_CMD(msg=False)

Create and solve an LSCP instance

In [ ]:
lscp = LSCP.from_cost_matrix(cost_matrix, SERVICE_RADIUS)
lscp = lscp.solve(solver)

How many first aid sites are needed to entirely cover the university buildings?

In [ ]:
lscp.problem.objective.value()

In [ ]:
len(lscp.fac2cli)

Add the results to the map

In [ ]:
sited_first_aid_stations_fg = folium.FeatureGroup("Sited First Aid Stations").add_to(m)
allocated_buildings_demand_fg = folium.FeatureGroup("Allocated Buildings").add_to(m)
solution = {}
colors = [
    "darkcyan",
    "mediumseagreen",
    "saddlebrown",
    "lightskyblue",
    "darkgoldenrod",
    "coral",
    "mediumvioletred",
    "blueviolet",
    "mediumorchid",
]
client_marker_size = 30
numpy.random.seed(0)

m = result.explore()
first_aid_stations_locations.explore(m=m, color="black", tooltip=False)

for i in range(len(first_aid_stations_locations)):
    if lscp.fac2cli[i]:
        color_choice_idx = numpy.random.choice(range(len(colors)))
        color = colors.pop(color_choice_idx)

        first_aid_stations_locations.iloc[[i]].explore(
            m=m,
            marker_kwds=dict(radius=client_marker_size),
            fill=True,
            color=color,
            marker_type="circle_marker"
        )

        # client locations with associated facility symbology
        client_marker_size -= 4
        result.iloc[lscp.fac2cli[i]].explore(m=m, color=color)
        solution[i] = []
        for j in lscp.fac2cli[i]:
            r = result.iloc[j]
            solution[i].append({"name": r["name"], "center": r.geometry.centroid.wkt, "idx": r.name})
m

It is ***important to note*** that the modeling solution is a direct result of the input cost matrix (seconds of travel) obtained from `routingpy.OSRM`. Although the solution set and decisions may not appear to optimal, the formulation of and solution to the LSCP are correct based on the input matrix.

In [ ]:
solution